In [9]:
from abc import ABC, abstractmethod

class MessageChannel(ABC):

    @abstractmethod
    def send(self, recipient: str, message: str) -> bool:
        pass

class EmailChannel(MessageChannel):

    def send(self, recipient: str, message: str) -> bool:
        print(f"[EMAIL] Отправка на {recipient}:\n{message}")
        return True

class SMSChannel(MessageChannel):

    def send(self, recipient: str, message: str) -> bool:
        print(f"[SMS] Отправка на номер {recipient}: {message}")
        return True

import itertools

class Message:
    _id_counter = itertools.count(1)

    def __init__(self, recipient: str, original_text: str, channel: MessageChannel):
        self.msg_id = next(self._id_counter)
        self.recipient = recipient
        self.original_text = original_text
        self.channel = channel

    @staticmethod
    def generate_id() -> int:
        return next(Message._id_counter)

    def send(self) -> bool:
        return self.channel.send(self.recipient, self.original_text)

from datetime import datetime

class MessageLogger:
    def __init__(self):
        self._log_entries = []

    def log(self, message: Message, success: bool):
        entry = {
            'timestamp': datetime.now(),
            'msg_id': message.msg_id,
            'recipient': message.recipient,
            'channel': message.channel.__class__.__name__,
            'original_text': message.original_text,
            'success': success
        }
        self._log_entries.append(entry)
        print(f"[LOG] Сообщение {message.msg_id} отправлено через {entry['channel']} -> Успех: {success}")

    def get_log(self):
        return self._log_entries.copy()

    def show_log(self):
        for entry in self._log_entries:
            print(f"{entry['timestamp']} | ID:{entry['msg_id']} | {entry['channel']} -> {entry['recipient']} | Успех:{entry['success']}")

class MessagingSystem:
    def __init__(self):
        self._channels = {}
        self._logger = MessageLogger()

    def add_channel(self, name: str, channel: MessageChannel):
        self._channels[name] = channel

    def send_message(self, recipient: str, text: str, channel_name: str) -> bool:
        channel = self._channels[channel_name]
        msg = Message(recipient, text, channel)
        success = msg.send()
        self._logger.log(msg, success)
        return success

    def get_logger(self):
        return self._logger

system = MessagingSystem()

email_channel = EmailChannel()
sms_channel = SMSChannel()
system.add_channel("email", email_channel)
system.add_channel("sms", sms_channel)

system.send_message("user@example.com",
                    "Ваш заказ подтверждён. Ожидайте доставку.",
                    "email")

print("\n--- Полный лог отправлений ---")
system.get_logger().show_log()

msg1 = Message("a@b.com", "Тест", email_channel)
msg2 = Message("c@d.com", "Тест2", sms_channel)
print(f"\nID сообщений: {msg1.msg_id}, {msg2.msg_id} (должны быть разными)")

[EMAIL] Отправка на user@example.com:
Ваш заказ подтверждён. Ожидайте доставку.
[LOG] Сообщение 1 отправлено через EmailChannel -> Успех: True

--- Полный лог отправлений ---
2026-05-18 09:26:57.770903 | ID:1 | EmailChannel -> user@example.com | Успех:True

ID сообщений: 2, 3 (должны быть разными)


In [27]:
# Создайте классы Engine и Car. Двигатель не может существовать без машины (композиция).
# У разных машин разные типы двигателей (бензиновый, электрический).
# Реализуйте статический метод для проверки валидности VIN-номера автомобиля.

from abc import ABC, abstractmethod
import re

class Engine(ABC):
    def __init__(self, car):
        self._car = car

    @abstractmethod
    def start(self):
        pass

    @abstractmethod
    def stop(self):
        pass

    @abstractmethod
    def get_type(self) -> str:
        pass

class BenzineEngine(Engine):
    def start(self):
        print(f"Бензиновый двигатель машины {self._car.get_vin()} запущен!")

    def stop(self):
        print(f"Бензиновый двигатель машины {self._car.get_vin()} остановлен!")

    def get_type(self) -> str:
        return f"Двигатель машины {self._car.get_vin()} - бензиновый!"

class ElectricEngine(Engine):
    def start(self):
        print(f"Электрический двигатель машины {self._car.get_vin()} запущен!")

    def stop(self):
        print(f"Электрический двигатель машины {self._car.get_vin()} остановлен!")

    def get_type(self) -> str:
        return f"Двигатель машины {self._car.get_vin()} - электрический!"

class Car:

    def __init__(self, vin: str, engine_type: str):
        self._vin = vin
        if engine_type.lower() == "бензиновый":
            self._engine = BenzineEngine(self)
        elif engine_type.lower() == "электрический":
            self._engine = ElectricEngine(self)
        else:
            raise ValueError("Неизвестный тип двигателя")

    def get_vin(self) -> str:
        return self._vin

    def start(self):
        self._engine.start()

    def get_engine_type(self) -> str:
        return self._engine.get_type()

    @staticmethod
    def valid_vin(vin: str) -> bool:
        if not isinstance(vin, str):
            return False
        pattern = r'^[A-HJ-NPR-Z0-9]{7}$'
        return bool(re.match(pattern, vin))

car1 = Car("1234AB1", "бензиновый")
car1.start()
print(f"Тип двигателя: {car1.get_engine_type()}")

car2 = Car("5678CD2", "электрический")
car2.start()
print(f"Тип двигателя: {car2.get_engine_type()}")

print("\nПроверка VIN '1234AB1':", Car.valid_vin("1234AB1"))

Бензиновый двигатель машины 1234AB1 запущен!
Тип двигателя: Двигатель машины 1234AB1 - бензиновый!
Электрический двигатель машины 5678CD2 запущен!
Тип двигателя: Двигатель машины 5678CD2 - электрический!

Проверка VIN '1234AB1': True


In [40]:
# Разные способы оплаты (Банковская карта, Электронные деньги) имеют свой расчет комиссии.
# Общий класс PaymentProcessor использует композицию с TaxCalculator, который вычисляет налог.
# Статический метод проверяет, является ли сумма допустимой.

from abc import ABC, abstractmethod

class PaymentMethod(ABC):
    @abstractmethod
    def calculate_fee(self, amount: float) -> float:
        pass

class CreditCard(PaymentMethod):
    def calculate_fee(self, amount: float) -> float:
        return max(5.0, amount * 0.035)

class ElectronicMoney(PaymentMethod):
    def calculate_fee(self, amount: float) -> float:
        return min(5000.0, amount * 0.005)

class TaxCalculator:
    tax_rate = 0.15

    def calculate_tax(self, amount: float) -> float:
        return amount * self.tax_rate

class PaymentProcessor:
    def __init__(self):
        self._tax_calculator = TaxCalculator()

    @staticmethod
    def valid_amount(amount: float) -> bool:
        return isinstance(amount, (int, float)) and 0 < amount <= 1000000

    def payment(self, amount: float, method: PaymentMethod) -> dict:
        if not self.valid_amount:
            raise ValueError(f"Недопустимая сумма {amount}!")
        fee = method.calculate_fee(amount)
        tax = self._tax_calculator.calculate_tax(amount)
        total = amount + fee + tax

        result = {
            "Начальная сумма": amount,
            "Комиссия": fee,
            "Налог": tax,
            "Итого": total,
            "Способ оплаты": method.__class__.__name__
        }
        return result

processor = PaymentProcessor()
card = CreditCard()
e_money = ElectronicMoney()
amount = 1000
print(f"\nПлатёж на сумму {amount} руб. картой:")
res = processor.payment(amount, card)
for k, v in res.items():
    print(f"  {k}: {v}")

print("\nПроверка суммы:", PaymentProcessor.valid_amount(-500))


Платёж на сумму 1000 руб. картой:
  Начальная сумма: 1000
  Комиссия: 35.0
  Налог: 150.0
  Итого: 1185.0
  Способ оплаты: CreditCard

Проверка суммы: False
